In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes seqeval evaluate
!pip install -U -q huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import load_dataset, DatasetDict, ClassLabel
from evaluate import load
from peft import get_peft_model, LoraConfig, TaskType
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR = "/content/drive/MyDrive/PLN"
DATA_DIR = f"{BASE_DIR}/data"
TRAINING_DIR = f"{BASE_DIR}/training"
MODEL_DIR = f"{BASE_DIR}/models"

In [ ]:
def parse_bio_file(path):
    sentences = []
    labels = []
    with open(path, encoding='utf-8') as f:
        words = []
        tags = []
        for line in f:
            line = line.strip()
            if not line:
                if words:
                    sentences.append(words)
                    labels.append(tags)
                    words = []
                    tags = []
                continue
            splits = line.split()
            if len(splits) >= 2:
                words.append(splits[0])
                tags.append(splits[-1])
    return {"tokens": sentences, "ner_tags": labels}


In [ ]:
from datasets import Dataset, DatasetDict


train_data = parse_bio_file(f"{DATA_DIR}/training.bio")
val_data = parse_bio_file(f"{DATA_DIR}/validation_cleaned.bio")
test_data = parse_bio_file(f"{DATA_DIR}/testing_cleaned.bio")

dataset = DatasetDict({
    "train": Dataset.from_dict(train_data),
    "validation": Dataset.from_dict(val_data),
    "test": Dataset.from_dict(test_data)
})


In [ ]:
label_names = list(set(tag for doc in train_data["ner_tags"] for tag in doc))

In [ ]:
unique_labels = sorted({tag for doc in dataset["train"]["ner_tags"] for tag in doc})
label2id = {l: i for i, l in enumerate(unique_labels)}
id2label = {i: l for l, i in label2id.items()}

def encode_tags(example):
    return {
        "ner_tags": [label2id[tag] if tag in label2id else -100 for tag in example["ner_tags"]]
    }

dataset = dataset.map(encode_tags)

Map:   0%|          | 0/3106 [00:00<?, ? examples/s]

Map:   0%|          | 0/929 [00:00<?, ? examples/s]

Map:   0%|          | 0/991 [00:00<?, ? examples/s]

In [ ]:
def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)
        elif word_id is None:
            new_labels.append(-100)
        else:
            label = labels[word_id]
            if label %2  == 1:
                label += 1
            new_labels.append(label)

    return new_labels

In [ ]:
model_name = "raulgdp/xml-roberta-large-finetuned-ner"
tokenizer = AutoTokenizer.from_pretrained(model_name, framework="pytorch")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
    )

    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Map:   0%|          | 0/3106 [00:00<?, ? examples/s]

Map:   0%|          | 0/929 [00:00<?, ? examples/s]

Map:   0%|          | 0/991 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
batch = data_collator([tokenized_dataset["train"][i] for i in range(2)])
batch["labels"]

tensor([[-100,   20,   20,   20,    4,    4,   14,   20,   20,   20,   20,   20,
           20,   20,   20, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100],
        [-100,   20,   20,   20,   20,   20,   20,   20,   20,   20,   20,   20,
      

In [ ]:
from transformers import AutoModelForTokenClassification

def model_init():
  model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

  peft_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none"
)

  model = get_peft_model(model, peft_config)
  return model

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
models = [
    {"epochs": 3, "batch_size": 4},
    {"epochs": 3, "batch_size": 8},
    {"epochs": 3, "batch_size": 16},
    {"epochs": 3, "batch_size": 32},
    {"epochs": 4, "batch_size": 4},
    {"epochs": 4, "batch_size": 8},
    {"epochs": 4, "batch_size": 16},
    {"epochs": 4, "batch_size": 32},
    {"epochs": 5, "batch_size": 4},
    {"epochs": 5, "batch_size": 8},
    {"epochs": 5, "batch_size": 16},
    {"epochs": 5, "batch_size": 32},
]

In [ ]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

In [ ]:
def save_model_hf(trainer, epochs, batch_size, eval_results):
    # Ruta para guardar el modelo
    save_dir = f"{MODEL_DIR}_ep{epochs}_bs{batch_size}"
    save_dir.mkdir(parents=True, exist_ok=True)

    # Guardar modelo y tokenizer
    trainer.save_model(str(save_dir))
    trainer.tokenizer.save_pretrained(str(save_dir))

    # Guardar resultados de evaluación en formato JSON
    eval_path = save_dir / "eval_results.json"
    with open(eval_path, "w") as f:
        json.dump(eval_results, f, indent=4)

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

for model_params in models:
    epochs = model_params["epochs"]
    batch_size = model_params["batch_size"]

    args = TrainingArguments(
        output_dir=TRAINING_DIR,
        save_strategy="epoch",
        learning_rate=2e-5,
        num_train_epochs=epochs,
        weight_decay=0.01,
        push_to_hub=False,
    )

    metric = load("seqeval")

    trainer = Trainer(
        model=model_init(),
        args=args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    results = trainer.evaluate()
    save_model_hf(trainer, epochs, batch_size, results)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-20-cc474f1c4f8b>:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForTokenClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss


TrainOutput(global_step=389, training_loss=2.047952921654081, metrics={'train_runtime': 1850.86, 'train_samples_per_second': 1.678, 'train_steps_per_second': 0.21, 'total_flos': 146999149664400.0, 'train_loss': 2.047952921654081, 'epoch': 1.0})